# Multi-Quarter Earnings & Thesis Reviewer

Zweite Variante neben `earnings_reviewer.ipynb`: deckt **alle Quartale seit dem Jahresbericht** ab (Scorecard je Quartal mit YTD und Trend, KPIs je Quartal, Thesen-Status je Quartal) und fuehrt ein **Thesen-Protokoll** (`reports/thesis_logs/`), das das naechste Update fortschreibt.

Kosten: Pressemitteilungen kostenlos (EDGAR); je 10-Q 1 sec-api-Call (MD&A), beim neuesten 2 (plus Risikofaktoren), danach Cache.

In [ ]:
# 1) Pfade und Imports
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]      # ...\FinRobot
AIF = ROOT / "ai_in_finance"
FR = ROOT / "finrobot_autogen"
sys.path.insert(0, str(AIF))

import autogen
from finrobot.utils import register_keys_from_json
from lib.earnings_reviewer_multi import run_multi_review
from lib.quarterly_data import multi_scorecard_text, load_thesis_log


In [ ]:
# 2) Modell und Schluessel
MODEL = "anthropic/claude-haiku-5.5"

base = json.load(open(FR / "OAI_CONFIG_LIST", encoding="utf-8"))
entry = next((e for e in base if e["model"] == MODEL), None) or dict(base[0], model=MODEL)
llm_config = {"config_list": [entry], "timeout": 300, "temperature": 0.3,
              "parallel_tool_calls": False, "max_tokens": 12000}
if MODEL.startswith("google/"):
    llm_config["extra_body"] = {"reasoning": {"effort": "low"}}
register_keys_from_json(str(FR / "config_api_keys"))


In [ ]:
# 3) Firmen und Geschaeftsjahr des Annual Reports (Ausgangsbasis); vorher den Annual Report Agent laufen lassen
COMPANIES = ["GOOG", "AMZN", "MA"]
FYEAR = "2025"
tag = MODEL.split("/")[-1]
save_dirs = {c: f"../reports/{c}_earnings_multi_{tag}" for c in COMPANIES}
for COMPANY in COMPANIES:                         # Vorschau ohne LLM: kommen Quartalsdaten, gibt es Thesen?
    print(f"\n{'=' * 30} {COMPANY} {'=' * 30}")
    try:
        print(multi_scorecard_text(COMPANY, FYEAR))
    except Exception as e:
        print(f"Keine Scorecard: {type(e).__name__}: {e}")
    print("Bisherige Thesen-Protokolle:", len(load_thesis_log(COMPANY)))

In [ ]:
# 4) Agent je Titel starten (ein Fehler bei einem Titel stoppt die anderen nicht)
status = {}
for COMPANY in COMPANIES:
    print(f"\n{'=' * 30} {COMPANY} {'=' * 30}")
    try:
        run_multi_review(llm_config, COMPANY, FYEAR, save_dirs[COMPANY])
        status[COMPANY] = "fertig"
    except Exception as e:
        status[COMPANY] = f"Fehler: {type(e).__name__}: {e}"
print("\nStatus:", status)

In [ ]:
# 5) PDF je Titel anzeigen
import io, glob
import pymupdf as fitz
from PIL import Image
for COMPANY in COMPANIES:
    print(f"\n{'=' * 30} {COMPANY} {'=' * 30}")
    pdf = sorted(glob.glob(f"{save_dirs[COMPANY]}/*_multi_quarter_update.pdf"), key=os.path.getmtime)
    if pdf:
        for page in fitz.open(pdf[-1]):
            display(Image.open(io.BytesIO(page.get_pixmap(dpi=100).tobytes("png"))))
    else:
        print("Kein PDF erzeugt - siehe Ausgabe von Zelle 4.")